# Cliche Sentence Creator

In this spractical we will fine tune a lightweight Tranformer model (DistilGPT2) with a list of Cliche Sentences.
* We will use Parameter-Efficient Fine-Tuning (PEFT) which lets you fine-tune a pre-trained transformer model without changing all of its parameters.
* More specifically, we will use LoRA fine-tuning which adds a small number of **trainable** parameters to selected layers.
* Accelerate will **run the code in a device-independent** way (i.e., you don't need to explicitly manage whether you run on CPU, NVIDIA GPU, etc.).

## 0) Let's check that all necessary packages are installed and available
You should see something like:
```
PyTorch: 2.14.0+cu130
Transformers: 5.17.0
PEFT: 0.21.0
PEFT import successful!
```

In [ ]:
import torch
import peft

print("PyTorch:", torch.__version__)

print("PEFT:", peft.__version__)

import transformers
print("Transformers:", transformers.__version__)

from peft import LoraConfig, TaskType, get_peft_model, PeftModel

print("PEFT import successful!")

## 1) Import the necessary libraries and choose CPU or NVIDIA GPU

Choose the device in the `DEVICE_PREFERENCE` setting below:

- `"auto"`: use an NVIDIA GPU (CUDA) when PyTorch can see one; otherwise use the CPU.
- `"cuda"`: require an NVIDIA GPU. The notebook raises a helpful error if CUDA is unavailable.
- `"cpu"`: force CPU execution.

To use an NVIDIA GPU, install a CUDA-enabled build of PyTorch that matches your system, enable a GPU runtime (for example, in your notebook environment), then set `DEVICE_PREFERENCE = "cuda"` or leave it as `"auto"`. A CUDA-capable NVIDIA driver alone is not sufficient if PyTorch was installed as a CPU-only build.

Run this cell before loading the model. If you change the preference, rerun this cell and then rerun the model, LoRA, optimizer, and training setup cells below.


In [ ]:
import os
import random
import numpy as np
import torch
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, TaskType, get_peft_model, PeftModel


# ---------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Seed CUDA as well when NVIDIA GPU support is available.
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ---------------------------------------------------------------
# Select the device used for both training and inference
# ---------------------------------------------------------------

DEVICE_PREFERENCE = "auto"  # Choose: "auto", "cpu", or "cuda"

if DEVICE_PREFERENCE not in {"auto", "cpu", "cuda"}:
    raise ValueError(
        "DEVICE_PREFERENCE must be 'auto', 'cpu', or 'cuda'."
    )

if DEVICE_PREFERENCE == "cuda":
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA was requested, but torch.cuda.is_available() is False. "
            "Check that this notebook has an NVIDIA GPU enabled and that "
            "a CUDA-enabled PyTorch build and compatible NVIDIA driver are installed."
        )
    device = torch.device("cuda")
elif DEVICE_PREFERENCE == "auto":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
else:
    device = torch.device("cpu")

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Selected device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(device))
    print("CUDA runtime reported by PyTorch:", torch.version.cuda)
else:
    print("Running on CPU.")


## 2) Now load the cliche senetences from the text file provided
*  You should see an output similar to: ```Number of lines in file: 245```

In [ ]:
FILE_PATH = os.getcwd()+"/cliche_sentences.txt"

# Read the file.
with open(FILE_PATH, "r", encoding="utf-8") as f:
    raw_sentences = [line.strip() for line in f if line.strip()]


print("Number of lines in file:", len(raw_sentences))

## 3) The text file contains a lot of duplicate sentences. Lets remove the duplicate lines and only keep the unique senetences and avoid unnecessary work during fine-tuning stage.

In [ ]:
# dict.fromkeys() removes duplicates while preserving the
# original order of the sentences.

sentences = list(dict.fromkeys(raw_sentences))

print("Number of unique sentences:", len(sentences))

# Display the first 10 examples
for i, sentence in enumerate(sentences[:10], start=1):
    print(f"{i}. {sentence}")

## 4) Load the DistilGPT2 transformer model from the Hugging Face Hub

- DistilGPT2 is a relatively small causal language model with about 82 million parameters.
- The same notebook code supports CPU and NVIDIA GPU execution; the device is selected in section 1.
- GPU training is generally faster, but still requires a CUDA-enabled PyTorch installation and an available NVIDIA GPU.
- The model weights require roughly 0.3 GB; additional memory is needed for the notebook, activations, and training.
- If you run out of memory, reduce `BATCH_SIZE` and/or the dataset `max_length`.
- You can safely ignore the warning that says `You are sending unauthenticated requests to the HF Hub`.
- You should see: `Model loaded successfully.`


In [ ]:
MODEL_NAME = "distilgpt2"

# Load the tokenizer.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# GPT models normally do not have a separate padding token.
# Use EOS as the padding token.
tokenizer.pad_token = tokenizer.eos_token

# Load the pre-trained model and move it to the selected device.
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.config.pad_token_id = tokenizer.pad_token_id
model = model.to(device)

print("Model loaded successfully on:", next(model.parameters()).device)


## 5) Prepare the cliche sentences for fine-tuning
* here we want each sentence to end with an End-Of-Sentence (EOS) token.

In [ ]:
# Add the EOS token to every training sentence.
#
# The EOS token tells the model that "This sentence ends here.".

training_texts = [
    sentence + tokenizer.eos_token
    for sentence in sentences
]

print(training_texts[0])

## 6) Convert the sentences to a PyTorch dataset
You should see an output similar to: ```Number of training examples: 38```

In [ ]:
class ClicheDataset(Dataset):
    """
    PyTorch Dataset containing the cliche sentences.

    Each sentence is converted into token IDs using
    the DistilGPT2 tokenizer.
    """
    def __init__(self, texts, tokenizer, max_length=64):

        self.examples = []

        for text in texts:

            # Convert text into integer token IDs.
            encoded = tokenizer(
                text,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            # Remove the extra batch dimension.
            input_ids = encoded["input_ids"].squeeze(0)

            self.examples.append(input_ids)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


dataset = ClicheDataset(
    training_texts,
    tokenizer,
    max_length=64
)

print("Number of training examples:", len(dataset))

## 7) Create a PyTorch DataLoader to load the dataset created above
* Since sentences have different lengths we need to pad sentences within each batch of data.
* You should see an output similar to: ```Number of batches: 10```

In [ ]:
BATCH_SIZE = 4

def collate_batch(batch):
    """
    Combine variable-length examples into one padded batch.
    """
    input_ids = pad_sequence(
        batch,
        batch_first=True,
        padding_value=tokenizer.pad_token_id
    )

    # 1 = real token; 0 = padding.
    attention_mask = (input_ids != tokenizer.pad_token_id).long()

    # Ignore padding positions when calculating the language-model loss.
    labels = input_ids.clone()
    labels[attention_mask == 0] = -100

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }


dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch,
    # Pinned host memory can make CPU-to-GPU transfers more efficient.
    pin_memory=(device.type == "cuda"),
    num_workers=0
)

print("Number of batches:", len(dataloader))


## 8) Configure LoRA (Low-Rank Adaptation) adapter
LoRA **adds small trainable matrices to selected parts of the model**, instead of changing **all** the parameters of the model.

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,

    # Rank of the LoRA matrices.
    # Smaller values use less memory.
    r=4,

    # Scaling factor for the LoRA update.
    lora_alpha=8,

    # Dropout applied to the LoRA layers.
    lora_dropout=0.05,

    # We are not training the original bias parameters.
    bias="none",

    # DistilGPT2 uses a module called c_attn for its attention projections.
    target_modules=["c_attn"]
)

## 9) Add the LoRA configuration to DistilGPT2 model
* You should see an output similar to: ```trainable params: 73,728 || all params: 81,986,304 || trainable%: 0.0899```
* Ignore the **UserWarning**:```fan_in_fan_out is set to False but the target module is .. ```

In [ ]:
# Wrap the pre-trained DistilGPT2 model with the LoRA adapters.
model = get_peft_model(
    model,
    lora_config
)

# Display how many parameters will actually be trained.
model.print_trainable_parameters()

## 10) Create the optimizer
* Optimiser should only update the parameters marked as trainable.

In [ ]:
LEARNING_RATE = 1e-4

optimizer = torch.optim.AdamW(
    filter(
        lambda parameter: parameter.requires_grad,
        model.parameters()
    ),
    lr=LEARNING_RATE
)

## 11) Configure the fine-tuning settings
* Because the dataset is **very small** we need several paases through it.

In [ ]:
NUM_EPOCHS = 40

loss_history = []

## 12) Fine-tune the model

- The training loop transfers each batch to the selected `device` (CPU or CUDA GPU).
- No other training-loop changes are needed when switching devices.
- You should see one loss value printed for each epoch.


In [ ]:
model.train()

for epoch in range(NUM_EPOCHS):

    total_loss = 0.0

    for batch in dataloader:

        # Transfer each batch to the same device as the model.
        # non_blocking is useful with pinned memory on CUDA.
        input_ids = batch["input_ids"].to(device, non_blocking=(device.type == "cuda"))
        attention_mask = batch["attention_mask"].to(device, non_blocking=(device.type == "cuda"))
        labels = batch["labels"].to(device, non_blocking=(device.type == "cuda"))

        optimizer.zero_grad()

        # Forward pass: the causal language model shifts labels internally
        # so that each token is trained to predict the next token.
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        # Backward pass and LoRA parameter update.
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()
        total_loss += loss.item()

    average_loss = total_loss / len(dataloader)
    loss_history.append(average_loss)

    print(
        f"Epoch {epoch + 1:02d}/{NUM_EPOCHS} "
        f"- Loss: {average_loss:.4f}"
    )


## 13) Plot the training loss

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    range(1, NUM_EPOCHS + 1),
    loss_history,
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("DistilGPT2 LoRA Fine-Tuning Loss")

plt.grid(True)
plt.show()

## 14) Switch the model to evaluation (inference) mode

The model remains on the selected device for inference. `model.eval()` changes layer behaviour (for example, dropout); it does not move the model between CPU and GPU.


In [ ]:
model.eval()

## 15) Create a function for text generation
* ```temperature``` controls the randomness of the text generation: typicaly values between 0.1 to 2.0 used (0.1 being very predictable).
* ```top_p``` limits the number of tokens (e.g., think of sample of words) that can participate in the next word prediction; generator takes the mostly likely tokens until their cumulative probability reaches about top_p.

In [ ]:
def generate_text(prompt, max_new_tokens=30, temperature=0.9, top_p=0.95):
    """
    Generate text using the fine-tuned DistilGPT2 model.

    The model and tokenized inputs are placed on the selected device.
    """
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    # Keep inference inputs on the same device as the model.
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Inference does not need gradient tracking.
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    generated_text = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return generated_text


## 16) Generate a few cliche-like sentences

Inference uses the same device selected in section 1. To switch device, rerun the device-selection cell and then reload/rebuild the model and rerun the required setup/training cells.


In [ ]:
prompts = [
    "A little patience",
    "Good fortune",
    "A careful mind",
    "The wiser path",
    "A kind word"
]

for prompt in prompts:

    print("=" * 70)

    print(
        generate_text(
            prompt,
            max_new_tokens=25,
            temperature=0.9,
            top_p=0.95
        )
    )